# RoadWatch AI — chạy hệ thống trên Google Colab
**Chuẩn bị (làm bằng trình duyệt, không cần Terminal):**
1. Trong Google Drive tạo thư mục `MyDrive/RoadWatch_AI/`.
2. Upload `roadwatch.zip` (dự án này) vào đó, và upload checkpoint YOLO 4 lớp vào `MyDrive/RoadWatch_AI/models/` rồi **đổi tên thành `model.pt`**.
3. Trong Colab → biểu tượng chìa khóa (Secrets) tạo: `ROADWATCH_OFFICER_PASSWORD` (bắt buộc), `NGROK_AUTHTOKEN` (bắt buộc), và tùy chọn `GEMINI_API_KEY`, `TELEGRAM_BOT_TOKEN`, `TELEGRAM_CHAT_ID`. Bật quyền truy cập cho notebook.
4. Runtime → Change runtime type → T4 GPU (nếu có), rồi chạy lần lượt từng ô.

> Dữ liệu (CSDL SQLite + ảnh) chạy trên đĩa cục bộ của Colab vì SQLite không ổn định trực tiếp trên Drive. **Chạy ô "Sao lưu lên Drive" trước khi đóng Colab.**

In [ ]:
from google.colab import drive
from pathlib import Path
import zipfile, shutil, os
drive.mount('/content/drive')
DRIVE = Path('/content/drive/MyDrive/RoadWatch_AI')
APP = Path('/content/roadwatch')
zip_path = DRIVE / 'roadwatch.zip'
assert zip_path.is_file(), 'Chưa thấy roadwatch.zip trong MyDrive/RoadWatch_AI'
if APP.exists():
    shutil.rmtree(APP)
with zipfile.ZipFile(zip_path) as z:
    z.extractall('/content')
if not APP.exists():   # zip có thể chứa thư mục gốc khác tên
    APP = next(p for p in Path('/content').iterdir() if (p / 'app.py').exists())
model_src = DRIVE / 'models' / 'model.pt'
assert model_src.is_file(), 'Chưa thấy models/model.pt trên Drive'
(APP / 'models').mkdir(exist_ok=True)
shutil.copy(model_src, APP / 'models' / 'model.pt')
print('Dự án:', APP)

In [ ]:
!pip -q install -r {APP}/requirements.txt pyngrok
# Nếu Colab báo cần khởi động lại runtime sau khi cài: Runtime → Restart session, rồi chạy lại từ ô đầu.

In [ ]:
# Kiểm tra checkpoint đúng 4 lớp D00/D10/D20/D40 (dừng nếu sai)
import sys
sys.path.insert(0, str(APP))
from core.detector import RoadDamageDetector
d = RoadDamageDetector(APP / 'models' / 'model.pt')
print('Ánh xạ lớp đã xác minh:', d.class_to_code)

In [ ]:
# Khôi phục dữ liệu từ bản sao lưu mới nhất trên Drive (nếu có)
DATA = Path('/content/roadwatch_data'); DATA.mkdir(exist_ok=True)
os.environ['ROADWATCH_DATA'] = str(DATA)
backups = sorted((DRIVE / 'backups').glob('roadwatch_backup_*.zip')) if (DRIVE / 'backups').exists() else []
if backups:
    from core.backup import restore_backup
    restore_backup(backups[-1].read_bytes())
    print('Đã khôi phục:', backups[-1].name)
else:
    print('Chưa có bản sao lưu; bắt đầu với dữ liệu trống.')

In [ ]:
# Nạp bí mật từ Colab Secrets
from google.colab import userdata
for key in ('ROADWATCH_OFFICER_PASSWORD', 'GEMINI_API_KEY', 'TELEGRAM_BOT_TOKEN', 'TELEGRAM_CHAT_ID'):
    try:
        v = userdata.get(key)
        if v: os.environ[key] = v
    except Exception:
        pass
assert os.environ.get('ROADWATCH_OFFICER_PASSWORD'), 'Hãy đặt ROADWATCH_OFFICER_PASSWORD trong Secrets (chức năng cán bộ bị khóa nếu thiếu).'
os.environ['ROADWATCH_MODEL'] = str(APP / 'models' / 'model.pt')
print('Đã cấu hình. Gemini:', bool(os.environ.get('GEMINI_API_KEY')), '| Telegram:', bool(os.environ.get('TELEGRAM_BOT_TOKEN')))

In [ ]:
import subprocess, time, urllib.request
from pyngrok import ngrok
ngrok.set_auth_token(userdata.get('NGROK_AUTHTOKEN'))
ngrok.kill()
try: streamlit_process.terminate()
except NameError: pass
link = ngrok.connect(8501, 'http')                      # mở đường hầm trước để biết URL
os.environ['ROADWATCH_BASE_URL'] = link.public_url      # dùng cho mã QR trang Chi tiết
log = open('/content/streamlit.log', 'w')
streamlit_process = subprocess.Popen(
    ['streamlit', 'run', 'app.py', '--server.port=8501', '--server.address=0.0.0.0', '--server.headless=true'],
    cwd=str(APP), env=os.environ.copy(), stdout=log, stderr=subprocess.STDOUT)
for _ in range(30):
    try:
        with urllib.request.urlopen('http://127.0.0.1:8501/_stcore/health', timeout=1) as r:
            if r.status == 200: break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError('Streamlit không khởi động; xem /content/streamlit.log')
print('MỞ ỨNG DỤNG:', link.public_url)

## Sao lưu lên Drive (chạy trước khi đóng Colab, và định kỳ khi demo)

In [ ]:
from datetime import datetime
from core.backup import make_backup
(DRIVE / 'backups').mkdir(exist_ok=True)
out = DRIVE / 'backups' / f"roadwatch_backup_{datetime.now():%Y%m%d_%H%M%S}.zip"
out.write_bytes(make_backup())
print('Đã lưu:', out)